# YouTube spot check

A small out-of-domain check of the three recognisers on the **Awaaz se Alfaaz YouTube Evaluation Set**
([`awaaz-se-alfaaz/YouTube-Evaluation-Set`](https://huggingface.co/datasets/awaaz-se-alfaaz/YouTube-Evaluation-Set)):
30 Urdu YouTube Shorts (news, sports, current affairs, often several speakers) with human gold transcripts.
The dataset also gives `model_transcript`, the output of the dataset authors' own system
(Whisper-large-v3-turbo fine-tuned with LoRA, then post-processed by Qwen3-14B). It is scored here as a fourth system.

The audio is not in the dataset. It is downloaded from YouTube with `yt-dlp`; clips that cannot be downloaded are listed and left out.

| Step | What is done |
|---|---|
| Audio | Downloaded with `yt-dlp`, converted to 16 kHz mono 32-bit float WAV the same way as experiment 01 (`soundfile` read, channel mean, `librosa.resample`) |
| Segments | Every clip is longer than 30 s. Each clip is cut once at pauses into pieces of at most 30 s, with the rule from Indic Transcribe's own `long_form.split_on_silence`. All three recognisers get the same pieces; each piece is transcribed in one pass and the texts are joined in order |
| Recognisers | Same models and decoding as experiment 01: Whisper-large-v3 (greedy, language Urdu), Omnilingual LLM 1B v2 (pipeline defaults), Indic Transcribe Core (`lang="ur"`) |
| Scoring | `clean_urdu` v1 on reference and transcript, corpus WER, CER, S / D / I %, paired bootstrap (1,000 resamples, seed 42), in the same file format as experiment 01 |

Runs locally (YouTube blocks most cloud machines). The notebook kernel is the Python 3.10 environment that has Omnilingual and Whisper (`omnienv`);
Indic Transcribe runs in its own environment (`indicenv`) as a separate process.

Outputs:

| File | Contents |
|---|---|
| `data/samples_youtube_eval.csv` | One row per video: `clip_id`, name, URL, downloaded or not (and why not), duration, segments, reference, the dataset's `model_transcript` |
| `data/segments.csv` | Start and end of every piece each clip was cut into |
| `results/transcripts/<recogniser>_transcripts.csv` | Joined transcript, seconds taken, status, decoding, the transcript of each piece |
| `results/transcripts/all_systems_by_clip.csv` | All four systems side by side, raw and normalised, with per-clip WER and CER |
| `results/scores/youtube_spot_check_scores.csv` | WER, CER, S / D / I %, time (same columns as `recogniser_scores.csv` in experiment 01) |
| `results/scores/youtube_spot_check_paired_bootstrap.csv` | WER difference between every pair of systems with its 95% interval (same columns as `paired_bootstrap.csv` in experiment 01) |
| `results/run_info/youtube_spot_check.json` | Machine, package versions, dataset revision, download results, segment settings, elapsed time |

Audio files are kept outside the repository (`~/hf_data/youtube_eval_audio`), because the videos belong to their channels.

## 1. Setup

In [ ]:
import os, sys, json, glob, time, re, string, unicodedata, subprocess
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf, librosa

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp"], check=True)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PROMPTS.md").exists())
EXP = REPO / "experiments" / "07_youtube_spot_check"
DATA, RESULTS = EXP / "data", EXP / "results"
for d in [DATA, RESULTS / "transcripts", RESULTS / "scores", RESULTS / "run_info"]:
    d.mkdir(parents=True, exist_ok=True)

AUDIO_DIR = Path.home() / "hf_data" / "youtube_eval_audio"      # 16 kHz WAVs, outside the repository
RAW_DIR = AUDIO_DIR / "downloaded"                                # files as yt-dlp saved them
RAW_DIR.mkdir(parents=True, exist_ok=True)
INDIC_PY = str(Path.home() / "indicenv" / "bin" / "python")       # environment with Indic Transcribe
INDIC_CACHE = str(Path.home() / "hf_data" / "hub")               # writable Hugging Face cache; the Indic model is already here

DATASET_REPO = "awaaz-se-alfaaz/YouTube-Evaluation-Set"
DATASET_REVISION = "6e4f7d595d165078094ee698e132b6573cebc706"   # dataset version used; pinned so the run can be repeated
SAMPLE_RATE = 16000
SEGMENT_TARGET_S, SEGMENT_MAX_S = 25.0, 30.0

SYSTEMS = {
    "whisper_large_v3": "Whisper-large-v3",
    "omnilingual_llm_1b_v2": "Omnilingual LLM 1B v2",
    "indic_transcribe_core": "Indic Transcribe Core",
    "awaaz_model_transcript": "Awaaz se Alfaaz system (dataset's model_transcript)",
}
RECOGNISERS = list(SYSTEMS)[:3]

# clean_urdu v1, copied unchanged from experiment 01
NORMALISER_VERSION = "clean_urdu v1"
_DIAC_RE = re.compile("[ؐ-ؚـً-ٰۖ-ۜ۟-۪ۨ-ۭ]")
_PUNCT = set(string.punctuation) | set("۔،؟؛٪।॥“”"
                                       "‘’—–…«»‖‐‒·")
_IZAFAT = {"اے", "ای", "ئے", "ء", "ے", "او"}
_UMAP = {"ي": "ی", "ك": "ک", "ة": "ہ", "ۃ": "ہ",
         "ى": "ی", "إ": "ا", "أ": "ا", "آ": "ا",
         "ٲ": "ا", "ٳ": "ا", "ؤ": "و", "ئ": "ی"}

def clean_urdu(text):
    t = unicodedata.normalize("NFC", str(text))
    t = _DIAC_RE.sub("", t)
    t = "".join(_UMAP.get(ch, ch) for ch in t)
    t = "".join(" " if ch in _PUNCT else ch for ch in t)
    return " ".join(w for w in t.split() if w not in _IZAFAT).strip()

print("yt-dlp", subprocess.run([sys.executable, "-m", "yt_dlp", "--version"], capture_output=True, text=True).stdout.strip())
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip())

## 2. Dataset and audio

The dataset CSV is read at the pinned revision. Five gold transcripts start with a byte-order mark (U+FEFF), which `clean_urdu` v1 would keep
as a word of its own, so it is replaced by a space when the CSV is read. Nothing else is changed.

Each video is downloaded once (over IPv4). A failed download is recorded with the last line of the `yt-dlp` message and the clip is left out of the recogniser scores.

In [ ]:
from huggingface_hub import hf_hub_download

src = pd.read_csv(hf_hub_download(DATASET_REPO, "awaaz_se_alfaaz_youtube_eval.csv", repo_type="dataset",
                                  revision=DATASET_REVISION, cache_dir=INDIC_CACHE), keep_default_na=False)
for col in ["gold_transcript", "model_transcript"]:
    src[col] = src[col].str.replace("﻿", " ", regex=False).str.strip()
assert src.video_id.is_unique


def download(video_id, url):
    # Returns (path, "") or (None, reason).
    found = glob.glob(str(RAW_DIR / f"{video_id}.*"))
    if found:
        return found[0], ""
    r = subprocess.run([sys.executable, "-m", "yt_dlp", "-4", "-f", "bestaudio/best", "--no-playlist",
                        "-o", str(RAW_DIR / f"{video_id}.%(ext)s"), url],
                       capture_output=True, text=True, timeout=600)
    found = glob.glob(str(RAW_DIR / f"{video_id}.*"))
    if found and r.returncode == 0:
        return found[0], ""
    lines = [l for l in (r.stderr + r.stdout).splitlines() if l.strip()]
    return None, (lines[-1] if lines else f"yt-dlp exit code {r.returncode}")[:300]


def to_16k_wav(raw_path, video_id):
    # Same steps as experiment 01: mono by channel mean, librosa resample to 16 kHz, 32-bit float WAV.
    out = AUDIO_DIR / f"{video_id}.wav"
    if not out.exists():
        tmp = AUDIO_DIR / f"{video_id}.decoded.wav"   # ffmpeg decodes the YouTube format at its own sample rate
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw_path, "-c:a", "pcm_f32le", str(tmp)], check=True)
        audio, sr = sf.read(tmp, dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != SAMPLE_RATE:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
        sf.write(out, audio, SAMPLE_RATE, subtype="FLOAT")
        tmp.unlink()
    return out, round(sf.info(out).frames / SAMPLE_RATE, 2)


rows = []
for r in src.itertuples():
    raw, reason = download(r.video_id, r.youtube_url)
    path, dur = to_16k_wav(raw, r.video_id) if raw else (None, np.nan)
    rows.append({"clip_id": f"youtube/{r.video_id}", "video_id": r.video_id, "video_name": r.video_name,
                 "youtube_url": r.youtube_url, "downloaded": raw is not None, "download_error": reason,
                 "duration_s": dur, "reference": r.gold_transcript, "model_transcript": r.model_transcript})
    print(f"{r.video_id:13s} {r.video_name[:30]:30s} " + (f"{dur:6.1f} s" if raw else f"FAILED: {reason}"), flush=True)
clips = pd.DataFrame(rows)
print(f"\nDownloaded {clips.downloaded.sum()} of {len(clips)} videos, {clips.duration_s.sum() / 60:.1f} minutes of audio")

### Cutting each clip into pieces of at most 30 s

Indic Transcribe Core trains on audio of at most 30 s and its model card says not to give it longer audio in one pass;
Omnilingual accepts at most 40 s; Whisper reads 30 s windows. So every clip is cut once, and all three recognisers get the same pieces.

The rule is Indic Transcribe's `long_form.split_on_silence`, rewritten here with NumPy so it does not depend on one recogniser's environment:
30 ms frames; a frame is quiet if it is more than 25 dB below the loudest frame; quiet runs of at least 0.25 s are possible cut points (their midpoint).
While more than 30 s remains, cut at the possible point closest to 25 s after the last cut, between 5 s and 30 s after it; if there is none, cut at 30 s.

In [ ]:
def split_on_silence(wav, sr=SAMPLE_RATE, target=SEGMENT_TARGET_S, hard_max=SEGMENT_MAX_S,
                     win_ms=30.0, rel_db=25.0, min_sil=0.25):
    n = len(wav)
    if n <= int(hard_max * sr):
        return [(0, n)]
    hop = max(1, int(sr * win_ms / 1000))
    frames = wav[: (n // hop) * hop].reshape(-1, hop).astype(np.float64)
    db = 20 * np.log10(np.sqrt(np.maximum((frames ** 2).mean(axis=1), 1e-12)))
    quiet = db < db.max() - rel_db
    runs, start = [], None
    for i, q in enumerate(quiet):
        if q and start is None:
            start = i
        elif not q and start is not None:
            runs.append((start, i)); start = None
    if start is not None:
        runs.append((start, len(quiet)))
    cuts = [((a + b) // 2) * hop for a, b in runs if (b - a) >= int(min_sil * sr / hop)]
    segs, pos = [], 0
    while n - pos > int(hard_max * sr):
        lo, hi, ideal = pos + int(5.0 * sr), pos + int(hard_max * sr), pos + int(target * sr)
        usable = [c for c in cuts if lo < c < hi]
        cut = min(usable, key=lambda c: abs(c - ideal)) if usable else hi
        segs.append((pos, cut)); pos = cut
    if pos < n:
        segs.append((pos, n))
    return segs


seg_rows = []
for c in clips[clips.downloaded].itertuples():
    wav = sf.read(AUDIO_DIR / f"{c.video_id}.wav", dtype="float32")[0]
    for k, (a, b) in enumerate(split_on_silence(wav)):
        seg_rows.append({"clip_id": c.clip_id, "segment": k, "start_sample": a, "end_sample": b,
                         "start_s": round(a / SAMPLE_RATE, 2), "end_s": round(b / SAMPLE_RATE, 2),
                         "length_s": round((b - a) / SAMPLE_RATE, 2)})
segments = pd.DataFrame(seg_rows)
segments.to_csv(DATA / "segments.csv", index=False, encoding="utf-8")

clips["segments"] = clips.clip_id.map(segments.groupby("clip_id").size()).fillna(0).astype(int)
clips.to_csv(DATA / "samples_youtube_eval.csv", index=False, encoding="utf-8-sig")
assert segments.length_s.max() <= SEGMENT_MAX_S
print(f"{len(segments)} pieces | length (s): min {segments.length_s.min()}, median {segments.length_s.median()}, max {segments.length_s.max()}")
print(clips.loc[clips.downloaded, ["duration_s", "segments"]].describe().round(1).to_string())

## 3. Transcription

Each recogniser runs as a separate process, so its GPU memory is freed when it finishes. A process reads `data/segments.csv`,
transcribes every piece of a clip in one pass, joins the pieces' texts with spaces and appends one row per clip to its transcripts file.
A clip already in the file is skipped, so a stopped run continues where it stopped.

A clip's status is `ok` if every piece is `ok`; otherwise it says how many pieces failed and the first reason
(an exception, an empty output, or for Whisper `cut_off`: the output stopped at the token limit instead of an end token).

In [ ]:
WORKER = r"""
import sys, os, json, time
import pandas as pd, soundfile as sf
EXP, AUDIO_DIR, STEM = sys.argv[1], sys.argv[2], sys.argv[3]
COLS = ["clip_id", "transcript", "time_s", "status", "decoding", "segments", "segment_transcripts"]
OUT = f"{EXP}/results/transcripts/{STEM}_transcripts.csv"

def run(one, decoding):
    seg = pd.read_csv(f"{EXP}/data/segments.csv")
    done = set(pd.read_csv(OUT).clip_id) if os.path.exists(OUT) else set()
    todo = [c for c in seg.clip_id.unique() if c not in done]
    print(f"{STEM}: {len(done)} clips done, {len(todo)} to go", flush=True)
    for k, cid in enumerate(todo, 1):
        audio = sf.read(f"{AUDIO_DIR}/{cid.split('/', 1)[1]}.wav", dtype="float32")[0]
        texts, statuses, t0 = [], [], time.perf_counter()
        for s in seg[seg.clip_id == cid].itertuples():
            try:
                text, status = one(audio[int(s.start_sample):int(s.end_sample)])
                text = str(text).strip()
                if not text:
                    status = "empty_output"
            except Exception as e:
                text, status = "", f"error: {type(e).__name__}: {e}"[:300]
            texts.append(text); statuses.append(status)
        bad = [x for x in statuses if x != "ok"]
        status = "ok" if not bad else f"{len(bad)} of {len(statuses)} pieces not ok: {bad[0]}"
        row = [cid, " ".join(t for t in texts if t), round(time.perf_counter() - t0, 3), status, decoding,
               len(texts), json.dumps(texts, ensure_ascii=False)]
        pd.DataFrame([row], columns=COLS).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False, encoding="utf-8")
        print(f"  {k}/{len(todo)} {cid} | {len(texts)} pieces | {row[2]:.1f} s | {status}", flush=True)
"""

WHISPER = WORKER + r"""
# Whisper is read from the shared Hugging Face cache, which is read-only here, so it is loaded offline.
os.environ["HF_HUB_OFFLINE"] = "1"
import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
name = "openai/whisper-large-v3"
proc = AutoProcessor.from_pretrained(name)
model = AutoModelForSpeechSeq2Seq.from_pretrained(name, torch_dtype=torch.float16, low_cpu_mem_usage=True).to("cuda").eval()
eos = model.generation_config.eos_token_id
beams = model.generation_config.num_beams or 1

def one(piece):
    feats = proc(piece, sampling_rate=16000, return_tensors="pt").input_features.to("cuda", torch.float16)
    with torch.no_grad():   # same call as experiment 01; with return_dict_in_generate the end token stays in the sequence
        out = model.generate(feats, language="urdu", task="transcribe", return_dict_in_generate=True, output_scores=True)
    seq = out.sequences[0]
    return proc.tokenizer.decode(seq, skip_special_tokens=True), ("ok" if seq[-1].item() == eos else "cut_off")

run(one, f"greedy, num_beams={beams}, one pass per piece")
"""

OMNILINGUAL = WORKER + r"""
# The Omnilingual checkpoint is already downloaded in ~/fairseq2_cache.
os.environ.setdefault("FAIRSEQ2_CACHE_DIR", os.path.expanduser("~/fairseq2_cache"))
import torch
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
pipe = ASRInferencePipeline(model_card="omniASR_LLM_1B_v2", device="cuda", dtype=torch.float16)

def one(piece):
    return pipe.transcribe([{"waveform": piece, "sample_rate": 16000}], lang=["urd_Arab"], batch_size=1)[0], "ok"

run(one, "pipeline defaults, one pass per piece")
"""

INDIC = WORKER + r"""
import torch
from huggingface_hub import snapshot_download
model_dir = snapshot_download("bodhan-ai/indic-transcribe-core", cache_dir=sys.argv[4], ignore_patterns=["nemo/*", "*.png"])
sys.path.insert(0, model_dir)
from indic_transcribe import IndicTranscribe
asr = IndicTranscribe.from_pretrained(model_dir)

def as_text(output):
    if isinstance(output, str):
        return output
    if isinstance(output, (list, tuple)):
        return as_text(output[0])
    if isinstance(output, dict):
        return output.get("text") or output.get("transcript") or str(output)
    return getattr(output, "text", str(output))

def one(piece):
    return as_text(asr(piece, lang="ur")), "ok"

run(one, "single pass per piece, model defaults")
"""


def run_worker(python, script, stem, *extra, env=None):
    # Runs one recogniser in its own process and prints its output as it comes.
    p = subprocess.Popen([python, "-u", "-c", script, str(EXP), str(AUDIO_DIR), stem, *extra],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    for line in p.stdout:
        if "it/s]" not in line and "Warning" not in line:
            print(line, end="", flush=True)
    assert p.wait() == 0, f"{stem} stopped with exit code {p.returncode}"

In [ ]:
run_worker(sys.executable, WHISPER, "whisper_large_v3")

In [ ]:
run_worker(sys.executable, OMNILINGUAL, "omnilingual_llm_1b_v2")

In [ ]:
run_worker(INDIC_PY, INDIC, "indic_transcribe_core", INDIC_CACHE)

## 4. Scores

Same scoring as experiment 01: `clean_urdu` v1 on both sides, corpus WER and CER, and S / D / I as % of reference words.
A failed or empty transcript counts every reference word as a deletion.

- **downloaded clips:** all four systems on the clips whose audio was downloaded.
- **all 30 clips:** only the dataset's `model_transcript`, which needs no audio. Added only if some downloads failed, to show whether the downloaded clips are typical.

In [ ]:
import jiwer

clips = pd.read_csv(DATA / "samples_youtube_eval.csv", keep_default_na=False)
clips["downloaded"] = clips.downloaded.astype(str) == "True"
clips["duration_s"] = pd.to_numeric(clips.duration_s, errors="coerce")
by_clip = clips[["clip_id", "video_name", "downloaded", "duration_s", "segments", "reference"]].copy()
by_clip["reference_clean"] = by_clip.reference.map(clean_urdu)

TRANSCRIPTS = {}
for stem in RECOGNISERS:
    t = pd.read_csv(RESULTS / "transcripts" / f"{stem}_transcripts.csv", keep_default_na=False).set_index("clip_id")
    missing = set(clips.clip_id[clips.downloaded]) - set(t.index)
    assert not missing, f"{SYSTEMS[stem]} has no transcript for {len(missing)} clips; run its step again"
    TRANSCRIPTS[stem] = t
    t = t.reindex(by_clip.clip_id)
    by_clip[stem] = t.transcript.fillna("").values
    by_clip[f"{stem}_time_s"] = pd.to_numeric(t.time_s, errors="coerce").values
    by_clip[f"{stem}_status"] = t.status.fillna("not downloaded").values
by_clip["awaaz_model_transcript"] = clips.model_transcript.values
by_clip["awaaz_model_transcript_time_s"] = np.nan
by_clip["awaaz_model_transcript_status"] = "ok"
for stem in SYSTEMS:
    by_clip[f"{stem}_clean"] = by_clip[stem].map(clean_urdu)


def edit_counts(ref, hyp):
    # Returns (S, D, I, reference words, character edits, reference characters).
    if not hyp:
        return 0, len(ref.split()), 0, len(ref.split()), len(ref), len(ref)
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return (w.substitutions, w.deletions, w.insertions, w.hits + w.substitutions + w.deletions,
            c.substitutions + c.deletions + c.insertions, c.hits + c.substitutions + c.deletions)


scorable = by_clip.reference_clean.str.len() > 0
has_text = {stem: by_clip.downloaded if stem in RECOGNISERS else pd.Series(True, index=by_clip.index) for stem in SYSTEMS}
COUNTS = {}
for stem in SYSTEMS:
    ok = scorable & has_text[stem]
    counts = np.array([edit_counts(r, h) if k else (0, 0, 0, 0, 0, 0)
                       for r, h, k in zip(by_clip.reference_clean, by_clip[f"{stem}_clean"], ok)])
    COUNTS[stem] = counts
    with np.errstate(invalid="ignore", divide="ignore"):
        by_clip[f"{stem}_WER_%"] = np.where(ok, np.round(100 * counts[:, :3].sum(1) / counts[:, 3], 2), np.nan)
        by_clip[f"{stem}_CER_%"] = np.where(ok, np.round(100 * counts[:, 4] / counts[:, 5], 2), np.nan)
by_clip.to_csv(RESULTS / "transcripts" / "all_systems_by_clip.csv", index=False, encoding="utf-8-sig")
print(f"Clips with an empty reference after normalisation (not scored): {(~scorable).sum()}")

In [ ]:
DATASET = "Awaaz se Alfaaz YouTube"
SUBSETS = [("downloaded clips", by_clip.downloaded & scorable, list(SYSTEMS))]
if not by_clip.downloaded.all():
    SUBSETS.append(("all 30 clips", scorable, ["awaaz_model_transcript"]))
reference_words = by_clip.reference_clean.str.split().str.len()
FILTERS = [("all clips", scorable), ("references with 3+ words", scorable & (reference_words >= 3))]

rows = []
for subset, in_subset, systems in SUBSETS:
    for filter_name, keep in FILTERS:
        m = (in_subset & keep).values
        for stem in systems:
            c = COUNTS[stem][m].sum(0)
            times = by_clip.loc[m, f"{stem}_time_s"]
            rows.append({"dataset": DATASET, "subset": subset, "clips_scored": filter_name, "clips": int(m.sum()),
                         "reference_words": int(c[3]), "recogniser": SYSTEMS[stem],
                         "WER_%": round(100 * c[:3].sum() / c[3], 2), "CER_%": round(100 * c[4] / c[5], 2),
                         "substitutions_%": round(100 * c[0] / c[3], 2), "deletions_%": round(100 * c[1] / c[3], 2),
                         "insertions_%": round(100 * c[2] / c[3], 2),
                         "failed_or_empty": int((by_clip.loc[m, f"{stem}_status"] != "ok").sum()),
                         "total_time_min": round(times.sum() / 60, 1) if times.notna().any() else np.nan,
                         "seconds_per_clip": round(times.mean(), 2) if times.notna().any() else np.nan})
scores = pd.DataFrame(rows)
scores.to_csv(RESULTS / "scores" / "youtube_spot_check_scores.csv", index=False, encoding="utf-8-sig")

show = ["recogniser", "clips", "WER_%", "CER_%", "substitutions_%", "deletions_%", "insertions_%",
        "failed_or_empty", "seconds_per_clip", "total_time_min"]
for (subset, filter_name), table in scores.groupby(["subset", "clips_scored"], sort=False):
    if filter_name == "all clips":
        print(f"\n{DATASET} | {subset} | {filter_name} | {int(table.reference_words.iloc[0]):,} reference words")
        print(table[show].to_string(index=False))

## 5. Paired bootstrap

For each pair of systems, on the same downloaded clips: draw as many clips as there are, with replacement; compute WER(A) − WER(B);
repeat 1,000 times (seed 42). The middle 95% of the differences is the 95% interval. With about 30 clips the intervals are wide, so
only large differences come out as reliable.

In [ ]:
from itertools import combinations

def paired_bootstrap(errors_a, errors_b, words, n=1000, seed=42):
    idx = np.random.default_rng(seed).integers(0, len(words), size=(n, len(words)))
    diff = 100 * (errors_a[idx].sum(1) - errors_b[idx].sum(1)) / words[idx].sum(1)
    return np.percentile(diff, [2.5, 97.5])

rows = []
for subset, in_subset, systems in SUBSETS:
    for filter_name, keep in FILTERS:
        m = (in_subset & keep).values
        for a, b in combinations(systems, 2):
            ea, eb = COUNTS[a][m, :3].sum(1), COUNTS[b][m, :3].sum(1)
            words = COUNTS[a][m, 3]
            lo, hi = paired_bootstrap(ea, eb, words)
            wa, wb = 100 * ea.sum() / words.sum(), 100 * eb.sum() / words.sum()
            verdict = (f"{SYSTEMS[a]} lower" if hi < 0 else f"{SYSTEMS[b]} lower" if lo > 0
                       else "no reliable difference")
            rows.append({"dataset": DATASET, "subset": subset, "clips_scored": filter_name, "clips": int(m.sum()),
                         "recogniser_A": SYSTEMS[a], "recogniser_B": SYSTEMS[b],
                         "WER_A_%": round(wa, 2), "WER_B_%": round(wb, 2), "difference_A_minus_B": round(wa - wb, 2),
                         "interval_low": round(lo, 2), "interval_high": round(hi, 2), "result": verdict})
boot = pd.DataFrame(rows)
boot.to_csv(RESULTS / "scores" / "youtube_spot_check_paired_bootstrap.csv", index=False, encoding="utf-8-sig")

show = ["recogniser_A", "recogniser_B", "WER_A_%", "WER_B_%", "difference_A_minus_B", "interval_low", "interval_high", "result"]
print(boot[(boot.subset == "downloaded clips") & (boot.clips_scored == "all clips")][show].to_string(index=False))

## 6. Per-clip WER and checks

- **Per clip:** WER of every system on every downloaded clip. A clip where even the best system is far off may mean the gold transcript
  does not match the audio that was downloaded (for example a re-uploaded or edited video).
- **Numbers:** `clean_urdu` v1 does not change numbers, so "38" and "اڑتیس" count as different words. The count of words with digits in them
  shows whether a system is penalised for writing numbers as digits where the gold transcript spells them out, or the reverse.

In [ ]:
d = by_clip[by_clip.downloaded].copy()
wer_cols = [f"{s}_WER_%" for s in SYSTEMS]
table = d[["video_name", "duration_s", "segments"] + wer_cols].rename(
    columns={f"{s}_WER_%": SYSTEMS[s].split(" (")[0] for s in SYSTEMS})
print(table.sort_values(table.columns[-2]).to_string(index=False))

best = d[wer_cols].min(axis=1)
print(f"\nClips where the best system has WER over 40% (possible reference mismatch): {(best > 40).sum()}")
for r in d[best > 40].itertuples():
    print(f"  {r.clip_id} {r.video_name}")

has_digit = lambda t: sum(any(ch.isdigit() for ch in w) for w in t.split())
print("\nWords containing digits, summed over downloaded clips:")
print(f"  {'gold reference':45s} {d.reference_clean.map(has_digit).sum()}")
for s in SYSTEMS:
    print(f"  {SYSTEMS[s]:45s} {d[f'{s}_clean'].map(has_digit).sum()}")

## 7. Example clips

In [ ]:
from IPython.display import HTML, display

style = "font-family:'Noto Nastaliq Urdu','Noto Naskh Arabic',serif;font-size:17px;direction:rtl;text-align:right;padding:2px 8px"
for _, r in by_clip[by_clip.downloaded].head(2).iterrows():
    html = f"<b>{r.clip_id}</b> {r.video_name} ({r.duration_s} s, {r.segments} pieces)"
    for label, col in [("Reference", "reference")] + [(n, s) for s, n in SYSTEMS.items()]:
        wer = "" if col == "reference" else f" — WER {r[f'{col}_WER_%']}%"
        html += f"<div style='font-size:11px;color:#777'>{label}{wer}</div><div style='{style}'>{r[col][:700]}</div>"
    display(HTML(html + "<hr>"))

## 8. Run information

In [ ]:
import torch, transformers, omnilingual_asr

indic_env = subprocess.run([INDIC_PY, "-c", "import sys, torch, transformers; print(sys.version.split()[0], torch.__version__, transformers.__version__)"],
                           capture_output=True, text=True).stdout.split()
gpu = subprocess.run("nvidia-smi --query-gpu=name,driver_version --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip().splitlines()[0]
yt_dlp_version = subprocess.run([sys.executable, "-m", "yt_dlp", "--version"], capture_output=True, text=True).stdout.strip()
ffmpeg_version = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.split("\n")[0]
info = {
    "machine": f"local, {gpu}",
    "dataset": {"repo": DATASET_REPO, "revision": DATASET_REVISION, "videos": len(clips),
                "reference": "gold_transcript (human-annotated); U+FEFF replaced by a space",
                "fourth_system": "model_transcript: Whisper-large-v3-turbo + LoRA, post-processed by Qwen3-14B (dataset card)"},
    "download": {"tool": f"yt-dlp {yt_dlp_version}", "ffmpeg": ffmpeg_version,
                 "downloaded": int(clips.downloaded.sum()), "failed": int((~clips.downloaded).sum()),
                 "failed_clips": clips.loc[~clips.downloaded, ["clip_id", "download_error"]].to_dict("records"),
                 "audio_minutes": round(float(clips.duration_s.sum()) / 60, 1)},
    "audio": "16 kHz mono 32-bit float WAV; channel mean, librosa.resample (as experiment 01)",
    "segments": {"rule": "Indic Transcribe long_form.split_on_silence, NumPy port", "target_s": SEGMENT_TARGET_S,
                 "max_s": SEGMENT_MAX_S, "frame_ms": 30, "quiet_below_loudest_db": 25, "min_quiet_s": 0.25,
                 "pieces": int(len(pd.read_csv(DATA / "segments.csv")))},
    "normaliser": NORMALISER_VERSION,
    "environments": {
        "omnienv (Omnilingual, Whisper)": {"python": sys.version.split()[0], "torch": torch.__version__,
                                           "transformers": transformers.__version__,
                                           "omnilingual_asr": getattr(omnilingual_asr, "__version__", "")},
        "indicenv (Indic Transcribe)": dict(zip(["python", "torch", "transformers"], indic_env)),
    },
    "models": {"whisper_large_v3": "openai/whisper-large-v3", "omnilingual_llm_1b_v2": "omniASR_LLM_1B_v2",
               "indic_transcribe_core": "bodhan-ai/indic-transcribe-core"},
    "decoding": {s: sorted(TRANSCRIPTS[s].decoding.unique().tolist()) for s in RECOGNISERS},
    "elapsed_minutes": {s: round(TRANSCRIPTS[s].time_s.astype(float).sum() / 60, 1) for s in RECOGNISERS},
    "bootstrap": {"resamples": 1000, "seed": 42},
}
json.dump(info, open(RESULTS / "run_info" / "youtube_spot_check.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(json.dumps(info, indent=2, ensure_ascii=False))